In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import KNNImputer
import os 

In [ ]:
def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Coluna não contém valores suficientes para análise.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.nan, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered


In [ ]:
def impute_knn(df, k=5):
    imputer = KNNImputer(n_neighbors=k)
    df_copy = df.copy()
    df_copy['Throughput'] = imputer.fit_transform(df_copy[['Throughput']])
    return df_copy

def impute_rolling_median(df, window_size=3):
    df_copy = df.copy()
    previous_na_count = df_copy['Throughput'].isna().sum()
    while df_copy['Throughput'].isna().any():
        df_copy['Throughput'] = df_copy['Throughput'].fillna(df_copy['Throughput'].rolling(window=window_size, center=True, min_periods=1).median())
        current_na_count = df_copy['Throughput'].isna().sum()
        if current_na_count >= previous_na_count:
            break  # No progress made, so exit the loop
        previous_na_count = current_na_count
    global_median = df_copy['Throughput'].median()
    df_copy['Throughput'] = df_copy['Throughput'].fillna(global_median)
    return df_copy

def impute_rolling_average(df, window_size=3):
    df_copy = df.copy()
    previous_na_count = df_copy['Throughput'].isna().sum()
    while df_copy['Throughput'].isna().any():
        df_copy['Throughput'] = df_copy['Throughput'].fillna(df_copy['Throughput'].rolling(window=window_size, center=True, min_periods=1).mean())
        current_na_count = df_copy['Throughput'].isna().sum()
        if current_na_count >= previous_na_count:
            break  # No progress made, so exit the loop
        previous_na_count = current_na_count
    global_mean = df_copy['Throughput'].mean()
    df_copy['Throughput'] = df_copy['Throughput'].fillna(global_mean)
    return df_copy

# def impute_rolling_median(df, window_size=3):
#     df_copy = df.copy()
#     while df_copy['Throughput'].isna().any():
#         df_copy['Throughput'] = df_copy['Throughput'].fillna(df_copy['Throughput'].rolling(window=window_size, center=True, min_periods=1).median())
#     return df_copy

# def impute_rolling_average(df, window_size=3):
#     df_copy = df.copy()
#     while df_copy['Throughput'].isna().any():
#         df_copy['Throughput'] = df_copy['Throughput'].fillna(df_copy['Throughput'].rolling(window=window_size, center=True, min_periods=1).mean())
#     return df_copy

# def impute_rolling_median(df, window_size=3):
#     df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, min_periods=1).median())
#     return df

# def impute_rolling_average(df, window_size=3):
#     df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, min_periods=1).mean())
#     return df

def linear_interpolation(df, limit_direction='both', order=1, method='linear'):
    df_final = df.copy()
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

    # Manter uma cópia da coluna 'Timestamp' antes de definir o índice
    timestamp_column = df['Timestamp'].copy()

    # Definir 'Timestamp' como índice
    df = df.set_index('Timestamp')

    # # Verificar se o índice é um DatetimeIndex
    # if not isinstance(df.index, pd.DatetimeIndex):
    #     print("O índice não é um DatetimeIndex. Verifique o formato da coluna 'Timestamp'.")
    #     return df  # Retorna o DataFrame original se a conversão falhar

    # Aplicar a interpolação
    if method in ['polynomial', 'spline']:
        if order is None:
            print(f"O parâmetro 'order' é necessário para o método '{method}'.")
            return df.reset_index()
        df_imputed = df.interpolate(method=method, order=order, limit_direction=limit_direction)
    else:
        df_imputed = df.interpolate(method=method, limit_direction=limit_direction)

    # Redefinir o índice para trazer 'Timestamp' de volta como coluna
    df_imputed = df_imputed.reset_index()

    # Restaurar a coluna 'Timestamp' original
    df_imputed['Timestamp'] = timestamp_column.values

    df_final['Throughput'] = df['Throughput'].combine_first(df_imputed['Throughput'])

    return df_imputed


In [ ]:
def decomposicao_svd(df):
    U, S, Vt = np.linalg.svd(df, full_matrices=True)
    return U, S, Vt

def grafico_variabilidade(variabilidade, S):
    plt.plot(range(1, len(variabilidade) + 1), variabilidade, marker='o', markersize=1, markerfacecolor='teal', markeredgecolor='teal', color='darkturquoise')
    plt.xlabel('Número de Valores Singulares')
    plt.ylabel('Variabilidade Acumulada')
    plt.title('Valores Singulares por Variabilidade Acumulada')
    plt.grid(color='lightgray', alpha=0.7)
    plt.show()

def componentes_principais(r, U, S, Vt):
    U_reduced = U[:, :r]
    S_reduced = S[:r]
    Vt_reduced = Vt[:r, :]
    return U_reduced, S_reduced, Vt_reduced
#trasnforma a matriz de volta em um dataframe
def matriztodf(dataframes):
    df = pd.DataFrame(dataframes)
    q = df.shape[0]*df.shape[1]
    df = df.transpose()
    df = df.to_numpy().reshape(-1, q)
    df = df.transpose()
    df = pd.DataFrame(df)
    if 0 in df.columns:
        df = df.rename(columns={0: 'Throughput'})
    return df

#calcula rmse dos valores gerados no svd com os valores 'originais'
def calcular_rmse(df1, df2, coluna):
    indices_comuns = df1.index.intersection(df2.index)
    valores_df1 = df1.loc[indices_comuns, coluna]
    valores_df2 = df2.loc[indices_comuns, coluna]
    rmse = np.sqrt(np.mean((valores_df1 - valores_df2) ** 2))
    return rmse

#gera arquivo csv final com a imputacao
def gerar_arq_csv(df1, df2, caminho_base, nome_arquivo_csv):
    df1['Throughput'] = pd.NA
    df1['Throughput'] = df2['Throughput']
    #exclui as ultimas linhas do arquivo que nao foi feita imputacao
    df1 = df1.dropna(subset=['Throughput'])
    caminho_svd = os.path.join(caminho_base, 'svd')
    if not os.path.exists(caminho_svd):
        os.makedirs(caminho_svd)
    caminho_arquivo_csv = os.path.join(caminho_svd, nome_arquivo_csv)
    # Salva o DataFrame resultante em um arquivo CSV
    df1.to_csv(caminho_arquivo_csv, index=False)
    print(f"CSV file '{caminho_arquivo_csv}' generated!")
    return df1

def matriz(path):
    df = pd.read_csv(path)
    df = outlier_removal(df, 'Throughput')
    df_datetime = df.copy()
    df_datetime.drop(columns=['Throughput'], inplace = True)
    df['Throughput'] = df['Throughput'].replace(-1, np.nan)
    
    Throughput = df['Throughput'].values
    num_dados = len(Throughput)
    num_colunas = num_dados // 28
    matriz = Throughput[:num_colunas*28].reshape(num_colunas, 28).T
    matriz_original = pd.DataFrame(matriz)
    df_interpolado = df["Throughput"].interpolate(method='linear', limit_direction='both')
    Throughput_=df_interpolado.values
    matriz_interpolado = Throughput_[:num_colunas*28].reshape(num_colunas, 28).T
    matriz_interpolado= pd.DataFrame(matriz_interpolado)
    mask = np.isnan(matriz_original.values)
    matriz_mascara = pd.DataFrame(mask)
    return matriz_original, matriz_mascara, matriz_interpolado, df_datetime#, r_max

In [ ]:
def get_dataset_path_list(diretory):
    dataset_path_list = []
    for file in os.listdir(diretory):
        path = os.path.join(diretory, file)
        dataset_path_list.append(path)
    return dataset_path_list

In [ ]:
def apply_basic_imputations(source_path, destination_path, csv_path_list):
    for caminho in csv_path_list:
        df = pd.read_csv(caminho)

        if (df.shape[0] < 28):
            print(f'The {caminho} file does not have sufficient quantity of lines for imputation (28)')
            continue

        df = outlier_removal(df, 'Throughput')
        
        df_knn = impute_knn(df.copy())
        
        df_rolling_median = impute_rolling_median(df.copy())
        
        df_rolling_average = impute_rolling_average(df.copy())

        df_interpolation = df.copy().interpolate(method='linear', limit_direction='both')

        # df_interpolation = linear_interpolation(df.copy())
        
        output_knn = caminho.replace(f'{source_path}', f'{destination_path}/knn/')
        output_median = caminho.replace(f'{source_path}', f'{destination_path}/mediana-movel/')
        output_average = caminho.replace(f'{source_path}', f'{destination_path}/media-movel/')
        output_interpolation = caminho.replace(f'{source_path}', f'{destination_path}/interpolacao-linear/')

        if not os.path.exists(f'{destination_path}/knn/'):
            os.makedirs(f'{destination_path}/knn/')
            print(f"Saving path for knn created.")
        if not os.path.exists(f'{destination_path}/mediana-movel/'):
            os.makedirs(f'{destination_path}/mediana-movel/')
            print(f"Saving path for mediana-movel created.")
        if not os.path.exists(f'{destination_path}/media-movel/'):
            os.makedirs(f'{destination_path}/media-movel/')
            print(f"Saving path for media-movel created.")
        if not os.path.exists(f'{destination_path}/interpolacao-linear/'):
            os.makedirs(f'{destination_path}/interpolacao-linear/')
            print(f"Saving path for interpolacao-linear created.")
        
        
        df_knn.to_csv(output_knn, index=False)
        df_rolling_median.to_csv(output_median, index=False)
        df_rolling_average.to_csv(output_average, index=False)
        df_interpolation.to_csv(output_interpolation, index=False)

        print(f"Processed file: {caminho}")

#adicionar uma verificação para caso o tamanho do arquivo seja menor que 28 -> remover porque se nao fica só interpolacao linear?
def apply_svd_imputation(destination_path, csv_path_list):
    
    resultados = {}

    for caminho_csv in csv_path_list:

        df = pd.read_csv(caminho_csv)

        if (df.shape[0] < 28):
            print(f'The {caminho_csv} file does not have sufficient quantity of lines for imputation (28)')
            continue

        nome_arquivo = os.path.basename(caminho_csv)
        
        resultados[nome_arquivo] = {'interpolacao_linear': None, 'svd_final': None}
        df_matriz, df_mask, df_interpolado, df_datetime = matriz(caminho_csv)
        resultados[nome_arquivo]['interpolacao_linear'] = df_interpolado.copy()

        A_anterior = df_interpolado.values.copy()
        rmse = float('inf') 
        max_iter = 300
        n_iter = 0


        while rmse >= 1e-3 and n_iter<=max_iter:  
            U, S, Vt = decomposicao_svd(df_interpolado)
            variabilidade = np.cumsum(S**2) / np.sum(S**2)

            porcentagem_variabilidade = 0.95
            r = np.where(variabilidade >= porcentagem_variabilidade)[0][0] + 1
            
            #print(f'Número de valores singulares para atingir {porcentagem_variabilidade*100}% de variabilidade: {r}')

            U_reduzido, S_reduzido, Vt_reduzido = componentes_principais(r, U, S, Vt)
            S_reduzido_matriz = np.diag(S_reduzido)

            A_aproximada = np.dot(np.dot(U_reduzido, S_reduzido_matriz), Vt_reduzido)
            A_aproximada_df = pd.DataFrame(A_aproximada)

            df_matriz_preenchida = df_matriz.fillna(A_aproximada_df)
            
            resultados[nome_arquivo]['svd_final'] = df_matriz_preenchida

            # Atualiza df_interpolado para a próxima iteração
            df_interpolado = df_matriz_preenchida.values
            
            # Calcular o RMSE entre a matriz atual e a anterior
            rmse = np.sqrt(np.mean((A_aproximada - A_anterior) ** 2))

            # Atualiza A_anterior para a próxima comparação
            A_anterior = A_aproximada.copy()

            n_iter +=1

        # print(f'RMSE na iteração atual: {rmse}')
        # print(f'Finalizando processamento para {caminho_csv}')

        svd = matriztodf(resultados[nome_arquivo]["svd_final"])
        interpolacao = matriztodf(resultados[nome_arquivo]["interpolacao_linear"])
        mask = matriztodf(df_mask)
        dfs_reshaped = []
        dfs_reshaped.append(interpolacao) #0
        dfs_reshaped.append(svd) #1
        dfs_reshaped.append(mask) #2
        # Chama a função para plotar os dados
        # plot_imputed_data(dfs_reshaped)
        gerar_arq_csv(df_datetime, dfs_reshaped[1], destination_path, nome_arquivo)
        

In [ ]:
def apply_all_imputations(source_path, destination_path):
    csv_path_list = get_dataset_path_list(source_path)
    apply_basic_imputations(source_path, destination_path, csv_path_list)
    apply_svd_imputation(destination_path, csv_path_list)

In [ ]:
longest_interval_with_failures = '../datasets/treated longest interval with failures/'
longest_interval_imputed = '../datasets/imputed-treated-longest-interval/'


In [ ]:
apply_all_imputations(longest_interval_with_failures, longest_interval_imputed)

In [ ]:
def verify_imputation(df1, df2, arquivo):
    # # 1. Verificar se a quantidade de colunas é igual
    # if df1.shape[1] == df2.shape[1]:
    #     print(f"A quantidade de colunas é igual. Arquivo: {arquivo}")
    # else:
    #     print(f"A quantidade de colunas é diferente. Arquivo: {arquivo}")

    # # 2. Verificar se a quantidade de linhas é igual
    # if df1.shape[0] == df2.shape[0]:
    #     print(f"A quantidade de linhas é igual. Arquivo: {arquivo}")
    # else:
    #     print(f"A quantidade de linhas é diferente. Arquivo: {arquivo}")

    # 3. Comparar a coluna 'Throughput'

    # a. Identificar onde df1['Throughput'] é NaN
    nan_indices = df1['Throughput'].isna()

    # Verificar se esses NaNs foram preenchidos em df2
    filled_in_df2 = df2['Throughput'][nan_indices].notna()

    if not filled_in_df2.all():
        print(f"Alguns valores NaN em df1['Throughput'] não foram preenchidos em df2. Arquivo: {arquivo}")

    # b. Verificar se os valores não NaN continuam os mesmos
    not_nan_indices = df1['Throughput'].notna()
    same_values = df1['Throughput'][not_nan_indices] == df2['Throughput'][not_nan_indices]

    if not same_values.all():
        print(f"Existem diferenças nos valores não NaN em 'Throughput' entre os datasets. Arquivo: {arquivo}")
        # Opcional: Mostrar as diferenças
        differences = df1.loc[not_nan_indices & ~same_values, ['Timestamp', 'Throughput']].assign(
            Throughput_df2=df2['Throughput']
        )
        print("Diferenças encontradas:")
        print(differences)

    df_merged = pd.merge(df1, df2, on='Timestamp', suffixes=('_df1', '_df2'))

    print(df_merged)


def compare_datasets_from_dir(diretorio1, diretorio2, tecnicas):
    for arquivo in os.listdir(diretorio1):
        path = os.path.join(diretorio1, arquivo)
        df = pd.read_csv(path)

        # Loop over each technique
        for tecnica in tecnicas:
            path2 = os.path.join(diretorio2, tecnica, arquivo)
            
            if not os.path.exists(path2):
                continue
            
            df2 = pd.read_csv(path2)
            
            verify_imputation(df, df2, arquivo)

In [ ]:
tecnicas = ['interpolacao-linear']

In [ ]:
compare_datasets_from_dir(longest_interval_with_failures, longest_interval_imputed, tecnicas)